# Video → trained humanoid policy

One clip in, one trained G1 policy out:
**SAM 3.1** tracks the athlete → MediaPipe 3D pose → mink retargets onto the G1 → **mjlab** trains a PPO motion-tracking policy in MuJoCo Warp → rollout video + metrics.

Runtime: **A100** (or L4 for a slower run). Put your clip in Drive at `a2h/clips/` (2–4 s, one athlete, full body, side view).
Secrets: `HF_TOKEN` (approved for `facebook/sam3.1`).

## 0. Install (once per runtime)

In [ ]:
REPO_URL = "https://github.com/PandaGrdn/athlete2humanoid.git"

from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, importlib.metadata
DRIVE_ROOT = "/content/drive/MyDrive/a2h"
REPO = "/content/athlete2humanoid"
for d in ("outputs", "clips"):
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)

numpy_before = importlib.metadata.version("numpy")
!pip install -q -r {REPO}/requirements.txt
if not os.path.exists("/content/sam3"):
    !git clone -q https://github.com/facebookresearch/sam3.git /content/sam3
!pip install -q -e "/content/sam3[notebooks]"
!bash {REPO}/scripts/fetch_assets.sh

# mjlab gets its own virtualenv: it needs newer torch/numpy than SAM 3.1, so they can't share one.
!pip install -q uv
if not os.path.exists("/content/mjlab_env"):
    !uv venv -q -p 3.12 /content/mjlab_env
!uv pip install -q -p /content/mjlab_env/bin/python mjlab==1.6.0 tensorboard

if importlib.metadata.version("numpy") != numpy_before:
    print(f"numpy changed {numpy_before} -> {importlib.metadata.version('numpy')}: restarting the runtime.")
    print("When it reconnects, skip this cell and run from step 1.")
    os.kill(os.getpid(), 9)

## 1. Session setup (after every restart)

In [ ]:
import os, sys, glob
os.environ["MUJOCO_GL"] = "egl"
DRIVE_ROOT = "/content/drive/MyDrive/a2h"
REPO = "/content/athlete2humanoid"
MJLAB_PY = "/content/mjlab_env/bin/python"
os.environ["A2H_OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs"
os.environ["A2H_CLIPS_DIR"] = f"{DRIVE_ROOT}/clips"
os.environ["A2H_ASSETS_DIR"] = f"{REPO}/assets"
os.chdir(REPO); sys.path.insert(0, REPO)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

import mediapy, numpy as np, torch, config
print("GPU:", torch.cuda.get_device_name(0))
print("clips:", sorted(os.path.basename(p) for p in glob.glob(f"{config.CLIPS_DIR}/*.mp4")))

## 2. Video → G1 reference motion
Trims the clip, tracks the athlete with SAM 3.1, lifts the pose to 3D, retargets it onto the G1.
Check the side-by-side before training: if the robot doesn't look like the athlete, the policy won't either.

In [ ]:
RAW = config.CLIPS_DIR / "raw.mp4"     # your clip
START, DURATION = 0.0, 4.0             # seconds to keep
NAME = "squat"                         # names the output folder
USE_SAM = True                         # False = whole frame (only for single-person clips)

CLIP = config.CLIPS_DIR / f"{NAME}.mp4"
!ffmpeg -y -loglevel error -ss {START} -t {DURATION} -i "{RAW}" -vf "fps=30,scale=-2:720" -an "{CLIP}"

import pipeline
result = pipeline.video_to_motion(CLIP, use_sam=USE_SAM)
OUT = result["out_dir"]
mediapy.show_video(mediapy.read_video(str(OUT / "side_by_side.mp4")), fps=30)

## 3. CSV → mjlab motion (forward kinematics at 50 fps)

In [ ]:
NPZ = OUT / "motion.npz"
!cd /tmp && MUJOCO_GL=egl WANDB_MODE=disabled {MJLAB_PY} -m mjlab.scripts.csv_to_npz --input-file "{OUT}/motion.csv" --output-name {NAME} --input-fps {result['fps']} --output-fps 50
!cp /tmp/motion.npz "{NPZ}"
print({k: v.shape for k, v in np.load(NPZ).items()})

## 4. Train the tracking policy (PPO, 4096 parallel robots)
Watch the iteration time in the first minute and set `ITERS` so it fits your budget.
mjlab's default is 30,000; a few thousand usually gives a first policy that can follow a short clip.
Checkpoints go to Drive, so a disconnect doesn't lose them.

In [ ]:
ITERS = 3000
LOG_ROOT = f"{DRIVE_ROOT}/logs/{NAME}"
!cd /tmp && MUJOCO_GL=egl WANDB_MODE=disabled /content/mjlab_env/bin/train Mjlab-Tracking-Flat-Unitree-G1 --env.commands.motion.motion-file "{NPZ}" --env.scene.num-envs 4096 --agent.max-iterations {ITERS} --agent.save-interval 500 --agent.logger tensorboard --log-root "{LOG_ROOT}"

## 5. Roll out the trained policy

In [ ]:
ckpts = sorted(glob.glob(f"{LOG_ROOT}/g1_tracking/*/model_*.pt"), key=lambda p: (os.path.getmtime(p), p))
CKPT = ckpts[-1]
print("checkpoint:", CKPT)
!MUJOCO_GL=egl {MJLAB_PY} {REPO}/mjlab_rollout.py --checkpoint "{CKPT}" --motion "{NPZ}" --out "{OUT}"
mediapy.show_video(mediapy.read_video(str(OUT / "policy.mp4")))

## Done when
- `side_by_side.mp4` shows the G1 copying the athlete (kinematic)
- `policy.mp4` shows the G1 doing it **with physics**, and `policy_metrics.json` has `falls: 0`

Everything lands in Drive at `a2h/outputs/<NAME>/` and `a2h/logs/<NAME>/`.